In [1]:
import os
import time

from rdmc import RDKitMol

from joblib import Parallel, delayed

from rdkit import Chem
import pandas as pd
import numpy as np

In [2]:
# paste the functions from utils.py
def canonicalize_smi(smi, remove_Hs=False, sanitize=True, remove_atom_mapping=False):
    """
    Create canonicalized SMILES
    
    """
    params = Chem.SmilesParserParams()
    params.removeHs = remove_Hs
    params.sanitize = sanitize

    # https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html#rdkit.Chem.rdmolfiles.MolFromSmiles
    # by default, sanitize = True
    # by default, removeHs = True darn
    mol = Chem.MolFromSmiles(smi, params)
    
    if not remove_Hs:
        mol = Chem.rdmolops.AddHs(mol)

    # Remove atom map numbers, otherwise the smiles string is long and non-readable
    if remove_atom_mapping:
        for atom in mol.GetAtoms():
            if atom.HasProp("molAtomMapNumber"):
                atom.ClearProp("molAtomMapNumber")
            # atom.SetAtomMapNum(0)
    
    return Chem.rdmolfiles.MolToSmiles(mol)

# modified from here
# https://github.com/rxn4chemistry/rxnfp/blob/6fd48f4927c2178555cc5d71dbfb225fb178f43c/rxnfp/tokenization.py#L122-L153
def process_reaction(rxn):
    """
    Process and canonicalize reaction SMILES
    """
    reactants, reagents, products = rxn.split(">")

    reactants_c = ".".join(sorted([canonicalize_smi(r, remove_atom_mapping=True) for r in reactants.split(".")]))

    if len(reagents) > 0:
        reagents_c = ".".join(sorted([canonicalize_smi(r, remove_atom_mapping=True) for r in reagents.split(".")]))
    else:
        reagents_c = ''

    products_c = ".".join(sorted([canonicalize_smi(p, remove_atom_mapping=True) for p in products.split(".")]))

    return f"{reactants_c}>{reagents_c}>{products_c}"

In [3]:
train_file = '../data/mlm_USPTO/original/mlm_train_file.txt'
eval_file = '../data/mlm_USPTO/original/mlm_eval_file_1k.txt'

### Do testing reactions

In [4]:
with open(eval_file, 'r') as f:
    lines = f.readlines()
    print(len(lines))
print(len(lines))

1000
1000


In [5]:
start = time.time()
new_smiles = Parallel(n_jobs=4)(delayed(process_reaction)(line) for line in lines)
print(f'Elapsed time: {time.time() - start:.2f} seconds')

Elapsed time: 0.91 seconds


In [ ]:
new_smiles[0]

In [10]:
with open('../data/mlm_USPTO/with_Hs/mlm_eval_file_Hs.txt', 'w') as f:
    f.write('\n'.join(new_smiles))

### Do training reactions

In [11]:
with open(train_file, 'r') as f:
    lines = f.readlines()
    print(len(lines))
print(len(lines))

801208
801208


In [13]:
start = time.time()
new_smiles = Parallel(n_jobs=4)(delayed(process_reaction)(line) for line in lines)
print(f'Elapsed time: {time.time() - start:.2f} seconds')

Elapsed time: 352.11 seconds


In [14]:
new_smiles

['[H]C1([H])OC([H])([H])C([H])([H])C1([H])[H].[H]OC([H])([H])C([H])([H])n1nc([H])c(-c2c([H])nc3c(c2[H])c(-c2c([H])nn(C([H])([H])c4c([H])c([H])c([H])c(F)c4[H])c2[H])c([H])n3S(=O)(=O)c2c([H])c([H])c(C([H])([H])[H])c([H])c2[H])c1[H].[H]OC([H])([H])[H].[H]O[H].[H][O-].[Li+]>>[H]OC([H])([H])C([H])([H])n1nc([H])c(-c2c([H])nc3c(c2[H])c(-c2c([H])nn(C([H])([H])c4c([H])c([H])c([H])c(F)c4[H])c2[H])c([H])n3[H])c1[H]',
 '[H]C([H])([H])[Mg]Br.[H]C1([H])OC([H])([H])C([H])([H])C1([H])[H].[H]c1c([H])c([H])c(C([H])([H])OC2([H])C([H])([H])C(=O)C([H])([H])C2([H])[H])c([H])c1[H]>>[H]OC1(C([H])([H])[H])C([H])([H])C([H])([H])C([H])(OC([H])([H])c2c([H])c([H])c([H])c([H])c2[H])C1([H])[H]',
 '[Cu]I.[H]C#CC([H])([H])C([H])(c1c([H])c([H])c([H])c(C([H])([H])[H])c1C([H])([H])[H])N([H])C(=O)OC(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H].[H]C([H])([H])C([H])([H])OC([H])([H])C([H])([H])[H].[H]N(C([H])([H])C([H])([H])[H])C([H])([H])C([H])([H])[H].[H]c1c([H])c(I)c([H])c(C([H])([H])[H])c1[H].[H]c1c([H])c([H])c([P](c2c(

In [15]:
with open('../data/mlm_USPTO/with_Hs/mlm_train_file_Hs.txt', 'w') as f:
    f.write('\n'.join(new_smiles))

In [7]:
params = Chem.SmilesParserParams()
params.removeHs = False
params.sanitize = True

In [8]:
def add_Hs(rxn_smiles):
    rsmi, psmi = rxn_smiles.split('>>')
    rmol = Chem.MolFromSmiles(rsmi, params) 
    rsmi = Chem.rdmolfiles.MolToSmiles(rmol)

    pmol = Chem.MolFromSmiles(psmi, params) 
    psmi = Chem.rdmolfiles.MolToSmiles(pmol)
    
    new_smiles = f'{rsmi}>>{psmi}'
    
    return new_smiles

In [9]:
with open(train_file, 'r') as f:
    lines = f.readlines()
    print(len(lines))
print(len(lines))

801208
801208


In [10]:
start = time.time()
new_smiles = Parallel(n_jobs=2)(delayed(add_Hs)(line) for line in lines[:100])
print(f'Elapsed time: {time.time() - start:.2f} seconds')

ArgumentError: Python argument types in
    rdkit.Chem.rdmolfiles.MolToSmiles(RDKitMol)
did not match C++ signature:
    MolToSmiles(RDKit::ROMol mol, bool isomericSmiles=True, bool kekuleSmiles=False, int rootedAtAtom=-1, bool canonical=True, bool allBondsExplicit=False, bool allHsExplicit=False, bool doRandom=False)

In [11]:
new_smiles

NameError: name 'new_smiles' is not defined

In [31]:
with open('mlm_train_file_Hs.txt', 'w') as f:
    f.write('\n'.join(new_smiles))

In [4]:
# write only the first 200,000 smiles from the training set
new_text = ''
with open('mlm_train_file_Hs.txt', 'r') as f:
    lines = f.readlines()
for line in lines[:200000]:
    new_text += line
with open('mlm_train_file_Hs_200k.txt', 'w') as f:
    f.write(new_text)

having the max sequence length be 256 may be a problem since adding Hs makes many reaction smiles over 400 characters long

In [32]:
with open(eval_file, 'r') as f:
    lines = f.readlines()
    print(len(lines))
print(len(lines))

1000
1000


In [33]:
start = time.time()
new_smiles = Parallel(n_jobs=4)(delayed(add_Hs)(line) for line in lines)
print(f'Elapsed time: {time.time() - start:.2f} seconds')

Elapsed time: 2.23 seconds


In [34]:
sum([len(smiles) > 400 for smiles in new_smiles])

415

In [35]:
with open('mlm_eval_file_1k_Hs.txt', 'w') as f:
    f.write('\n'.join(new_smiles))

In [16]:
new_smiles = []
count = 0
with open(eval_file, 'r') as f:
    lines = f.readlines()
    for rxn_smiles in lines:
        rsmi, psmi = rxn_smiles.split('>>')
        # print(rsmi)
        # print(psmi)
        
        rmol = RDKitMol.FromSmiles(rsmi)
        rsmi = rmol.ToSmiles(removeHs=False)
        
        pmol = RDKitMol.FromSmiles(psmi)
        psmi = pmol.ToSmiles(removeHs=False)
        new_smiles.append(f'{rsmi}>>{psmi}')
        if len(f'{rsmi}>>{psmi}') > 384:
            # print(len(f'{rsmi}>>{psmi}'))
            count += 1
count

458